# AquaLimit 실제 지역 분석 (Google Colab)

**California Aqueduct MP 195~215 (Lost Hills 주변 31km)** 실제 자료로

1. 지하수·침하 물리모델(MODFLOW 6 + CSUB, 층 구조는 USGS CVHM2)을 만들고
2. 위성(InSAR)·관측정으로 땅속 성질을 역산하고, 보정에 쓰지 않은 **GPS로 예측을 검증**하고
3. 침하 기준을 지키는 **취수 계획(E1~E4)**을 세우고
4. 물리모델을 흉내 내는 **AI 대리모델**을 학습합니다.

| 방식 | 보정 자료 |
|---|---|
| E1 | 없음(우물 용량 비례 배분) |
| E2 | 관측정 수위만 |
| E3 | 관측정 수위 + 위성(InSAR) |
| E4 | E3 + 불확실성(앙상블) |

**실행 방법:** 메뉴 **런타임 → 모두 실행**.
- 기본(`RECOMPUTE = False`): 저장된 결과를 불러와 표·그림을 보여줍니다. 약 3분.
- `RECOMPUTE = True`: 처음부터 다시 계산합니다. Colab 기본(CPU 2개)에서 약 2~3시간.

In [ ]:
RECOMPUTE = False      # True로 바꾸면 역산·계획·AI를 처음부터 다시 계산(약 2~3시간)

## 1. 설치

- `flopy`: Python으로 MODFLOW 모델을 만들고 실행하는 라이브러리
- `mf6`: USGS MODFLOW 6 실행파일 (`get-modflow`가 내려받음)
- `pyproj`, `shapely`: 좌표 변환, 지도 도형 / `fonts-nanum`: 그래프 한글 글꼴

In [ ]:
!pip -q install flopy shapely pyproj
!get-modflow /usr/local/bin --subset mf6 --quiet
!apt-get -qq install -y fonts-nanum > /dev/null
!mf6 --version

## 2. 코드와 자료 가져오기

GitHub 저장소에서 가져옵니다(`real/` 폴더를 커밋·푸시한 뒤 실행). 푸시하지 않았다면 왼쪽 파일 패널에 `real` 폴더를 통째로 올리고 `%cd /content/real`만 실행하세요.

In [ ]:
import os
if not os.path.exists("/content/AquaLimit"):
    !git clone -q https://github.com/seoyoooooni/AquaLimit.git /content/AquaLimit
%cd /content/AquaLimit/real
!ls

## 3. 자료 불러오기

| 자료 | 내용 |
|---|---|
| InSAR | DWR TRE Altamira 측정점 74,944개 → 1km 격자 월별 누적 침하(유전 제외). 2023년 이후는 GPS와 어긋나 쓰지 않음 |
| 관측정 | DWR 정기 측정 30곳, 관측정별 첫 측정 대비 수위 변화 |
| GPS | 수로 GPS 5곳(P544, Q198, Q204, Q208, Q213) — **검증 전용** |
| 강수 | NOAA Lost Hills·Buttonwillow 관측소 → 함양 |
| CVHM2 | USGS 센트럴밸리 모델에서 뽑은 층 구조·조립질 비율·압축 점토 두께(`data/cvhm2_box.npz`, `fetch_cvhm2.py`로 생성) |

In [ ]:
import os, time, json, warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display
import real_model as r
import run_real as rr          # 한글 글꼴 설정 포함

site = r.load_site()
print("모델 구조:", r.STRUCTURE, "/ 파라미터", len(r.NAMES), "개")
print("모델 격자:", site["ny"], "×", site["nx"], "(1km), 농경지 격자", int(site["ag"].sum()))
for p_ in ["cal", "val"]:
    i = r.obs_index(site, p_, True)
    print(f"{'보정(2015.7~2022.12)' if p_=='cal' else '검증(2023.1~2026.4)'}: InSAR {i['n_ins']:,}개, 수위 {i['n_hh']}개, GPS {len(i['gp'])}개")
cv = site["cv"]
for g in ["G1", "G2", "G3", "G4"]:
    print(f"  CVHM2 {g}: 두께 {np.median(cv[g+'_thk']):.0f}m, 조립질 {np.median(cv[g+'_pc'])*100:.0f}%, 압축 점토 {np.median(cv[g+'_fine']):.0f}m")

## 4. 모델 한 번 돌려보기

4개 층 묶음: 상부 대수층 / 코코란 점토 / 하부 대수층 위 / 하부 대수층 아래. 압축층은 **과거 최저 수위보다 내려가면 비탄성(영구) 압축**(CVHM2 계수 4.4e-4/m), 그 위에서는 탄성 압축(2.5e-5/m).

In [ ]:
t = time.time()
S, H = r.run_model(site, r.PRIOR_MEAN)
print(f"MODFLOW 1회 실행: {time.time()-t:.1f}초")
plt.figure(figsize=(5,4)); plt.imshow(S[-1], origin="lower", cmap="RdBu_r", vmin=-60, vmax=300)
plt.colorbar(label="누적 침하 (mm)"); plt.title("보정 전(사전값) 2026.4 누적 침하"); plt.show()

## 5. 역산 + GPS 검증 (E2·E3·E4)

**가우스-뉴턴 최소제곱 역산**으로 15개 파라미터(구역별 압축성 보정 배수 9개, 조립질·세립질·코코란 투수계수, 농경지 취수량, 가뭄 계수, 함양 비율)를 추정합니다.
E4 범위에는 **되돌아보기 검증**(2019년까지로 보정 → 2020~2022 예측 오차 측정)으로 잰 모델 구조 오차를 더합니다.

In [ ]:
if RECOMPUTE:
    rr.main()                   # 약 1시간 (역산 결과는 results/calib_cache.json에 저장되어 다시 실행하면 이어서 계산)
display(Markdown((rr.OUT / "RESULTS.md").read_text(encoding="utf-8")))

In [ ]:
display(Image(str(rr.OUT / "real-3-gps-validation.png")))
display(Image(str(rr.OUT / "real-4-map-fit.png")))

## 6. 취수 계획 (2026년 5~7월)

수로 동쪽 3km 가상 사업자 우물 6개, 3개월 수요 90만 m³, 기준: 계획으로 인한 수로 추가 침하 ≤ 2mm/3개월.
각 방식의 계획을 E4 앙상블 30개 모두에서 계산해 기준 초과 확률로 비교합니다.

In [ ]:
import run_plan as rp
if RECOMPUTE:
    rp.main()                   # 약 20분
display(Markdown((rp.OUT / "RESULTS.md").read_text(encoding="utf-8")))

## 7. AI 대리모델

MODFLOW 600회 결과로 신경망(21 → 256 → 256 → 76)을 학습해, 같은 계산을 1ms 안에 합니다. 앙상블 1,000개 강건 계획에 쓰고, 최종 계획은 MODFLOW로 다시 확인합니다.

In [ ]:
import surrogate_real as sr
if RECOMPUTE:
    sr.main()                   # 학습 자료 생성 약 35~60분 + 학습 수 분 (train_data.npz가 있으면 학습만)
display(Markdown((sr.OUT / "RESULTS.md").read_text(encoding="utf-8")))

## 8. 직접 해보기 — 수요와 기준을 바꿔 계획 세우기

미리 계산한 응답 행렬로 바로 계획을 세웁니다(MODFLOW를 다시 돌리지 않음).

In [ ]:
import plan_real as p
z4 = np.load(rp.OUT / "R4.npz"); R4 = [{"Rs": z4["Rs"][i], "Rh": z4["Rh"][i]} for i in range(len(z4["Rs"]))]
z3 = np.load(rp.OUT / "R3.npz"); R3 = {k: z3[k] for k in z3.files}

DEMAND_THOUSAND_M3 = 900     # 3개월 필요 수량(천 m³)
LIMIT_MM = 2.0               # 수로 추가 침하 기준(mm/3개월)

p.DEMAND, p.LIMIT_MM = DEMAND_THOUSAND_M3 * 1000.0, LIMIT_MM
rows = []
for name, plan in [("E1 기존 방식", p.baseline()), ("E3 추정값 1개", p.solve([R3])), ("E4 앙상블 30개", p.solve(R4))]:
    if not plan["feasible"]:
        rows.append({"방식": name, "결과": "실행 불가 — " + plan["reason"]}); continue
    rk = p.risk(plan, R4)
    rows.append({"방식": name, "비용(천$)": round(plan["cost"]/1e3), "외부 구매(천 m³)": round(plan["purchase"]/1e3),
                 "기준 초과 확률": f"{rk['exceed_prob']*100:.0f}%", **{f"우물 {chr(65+i)}": round(v) for i, v in enumerate(plan["rates"])}})
pd.DataFrame(rows)

## 9. 결과 내려받기

In [ ]:
!cd /content/AquaLimit/real && zip -qr results_all.zip results results_plan results_ai
try:
    from google.colab import files; files.download("/content/AquaLimit/real/results_all.zip")
except ImportError:
    print("Colab이 아니면 real/results_all.zip을 직접 가져가세요.")

---
**데모 화면**: 컴퓨터에서 `cd real && pip install streamlit plotly && streamlit run app_real.py`